#ENCODE THE OUTCOME VARIABLE & JOIN/MERGE WITH DEPRIVATION IMD SCORES

In [2]:
# encode the last outcome category into binary resolved/unresolved outcomes

STEPS TO TAKE

1. Read in the cleaned dataframe - saved as parquet this time
2. create copy of df for multinomial modelling later
3. remove ambiguous categories that are hard to justify
4. check 

In [3]:
#import libraries

import pandas as pd

In [4]:
# read in the dataframe

cleaned_df = pd.read_parquet("../data/cleaned_df.parquet")

### Notes for ref.
#### cleaned_df -the CSV
#### cleaned data - the parquet
#### model_df_multiclass - the original non collapsed outcome category df
#### model_df_binary - the df with only 2 binary options in outcome

In [5]:

# Take a copy of the df before collapsing to keep all original outcome categories for multinomial model later 

model_df_multiclass = cleaned_df.copy()

In [6]:
# realise this is a bit redundant but i'm making a copy anyway because I forget how to rename the df
model_df_binary = cleaned_df.copy()

The next step below creates a new column outcome_binary (1 = resolved, 0 = unresolved) while leaving last_outcome_category untouched as the original text 

In [7]:
# encode a binary version of the outcome category - using model_df_binary and outcome_binary is the new category

# Final classification, per the project's outcome category grouping table:
#   Resolved (1)   -> formal sanction or discretionary disposal by police
#   Unresolved (0) -> no actionable outcome for the victim
#   Excluded       -> pending/missing data, not a true outcome (dropped before modelling)

resolved_outcomes = [
    'Offender given a caution',
    'Suspect charged as part of another case',
    'Offender given a drugs possession warning',
    'Local resolution',
    'Action to be taken by another organisation',
    'Further action is not in the public interest',
    'Further investigation is not in the public interest',
    'Formal action is not in the public interest',
]

unresolved_outcomes = [
    'Unable to prosecute suspect',
    'Investigation complete; no suspect identified',
]

excluded_outcomes = [
    'Status update unavailable',
    'Awaiting court outcome',
    'Court result unavailable',
    'Under investigation',
]

# Drop excluded/pending categories BEFORE encoding, so they don't silently
# fall into the unresolved (0) class
model_df_binary = model_df_binary[
    ~model_df_binary['last_outcome_category'].isin(excluded_outcomes)
].copy()

# Sanity check: every remaining row should be classifiable as resolved or unresolved
unclassified = ~model_df_binary['last_outcome_category'].isin(resolved_outcomes + unresolved_outcomes)
if unclassified.any():
    print("Warning — unclassified categories found, review before proceeding:")
    print(model_df_binary.loc[unclassified, 'last_outcome_category'].value_counts())

model_df_binary['outcome_binary'] = model_df_binary['last_outcome_category'].apply(
    lambda x: 1 if x in resolved_outcomes else 0
)

# sanity check
print(model_df_binary['outcome_binary'].value_counts())
print(model_df_binary['outcome_binary'].value_counts(normalize=True).round(4) * 100)

outcome_binary
0    210054
1     17504
Name: count, dtype: int64
outcome_binary
0    92.31
1     7.69
Name: proportion, dtype: float64


In [8]:
# sanity check again to look at top few rows

model_df_binary.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,outcome_binary
1,03b3bda6de7c51f6cd0820b7f998a1f878e7acbb6b18dd...,2025-01,West Yorkshire Police,-1.882755,53.924927,On or near Moorside Lane,E01010646,Bradford 001A,Burglary,Investigation complete; no suspect identified,0
2,6f9fe9a78c7ed106fadb658e4fd5313c1bb5bc5c8aa223...,2025-01,West Yorkshire Police,-1.878940,53.943744,On or near Cross End Fold,E01010646,Bradford 001A,Shoplifting,Investigation complete; no suspect identified,0
3,9cbc69a540485581b55d8d4723eb41b97d364bc98132d8...,2025-01,West Yorkshire Police,-1.872959,53.941733,On or near Cornerstones Close,E01010646,Bradford 001A,Vehicle crime,Investigation complete; no suspect identified,0
4,050a7cc33129f37f41cbbd38fe858e63e07dc239720be8...,2025-01,West Yorkshire Police,-1.883567,53.934131,On or near Cocking Lane,E01010646,Bradford 001A,Violence and sexual offences,Unable to prosecute suspect,0
5,b43bb9c614a189dd550a9e877b409cdb6449d6ec1c1073...,2025-01,West Yorkshire Police,-1.880138,53.945417,On or near Aynholme Drive,E01010646,Bradford 001A,Violence and sexual offences,Investigation complete; no suspect identified,0


In [9]:
#confirm the encoding logic actually mapped correctly. This should show category by category, which text label got mapped to 0 vs 1 

model_df_binary.groupby('last_outcome_category')['outcome_binary'].first()

last_outcome_category
Action to be taken by another organisation             1
Formal action is not in the public interest            1
Further action is not in the public interest           1
Further investigation is not in the public interest    1
Investigation complete; no suspect identified          0
Local resolution                                       1
Offender given a caution                               1
Offender given a drugs possession warning              1
Suspect charged as part of another case                1
Unable to prosecute suspect                            0
Name: outcome_binary, dtype: int64

In [10]:
# value counts again alongside percentage 


counts = model_df_binary['last_outcome_category'].value_counts()
pct = model_df_binary['last_outcome_category'].value_counts(normalize=True) * 100
pd.DataFrame({'count': counts, 'pct': pct.round(2)})

,count,pct
last_outcome_category,,
Unable to prosecute suspect,109113,47.95
Investigation complete; no suspect identified,100941,44.36
Local resolution,6465,2.84
Further action is not in the public interest,3996,1.76
Action to be taken by another organisation,3667,1.61
Offender given a caution,1936,0.85
Further investigation is not in the public interest,917,0.40
Formal action is not in the public interest,499,0.22
Suspect charged as part of another case,23,0.01


In [11]:
model_df_binary.columns.tolist()

['crime_id',
 'month',
 'reported_by',
 'longitude',
 'latitude',
 'location',
 'lsoa_code',
 'lsoa_name',
 'crime_type',
 'last_outcome_category',
 'outcome_binary']

In [12]:
# MERGE/Join data to LSOA

In [13]:
#read in the IMD scores file

IMD5 = pd.read_csv('../data/raw/deprivation/File_5_IoD2025_ScoresFIOD.csv')

In [14]:
IMD5.columns.tolist()

['LSOA code (2021)',
 'LSOA name (2021)',
 'Local Authority District code (2024)',
 'Local Authority District name (2024)',
 'Index of Multiple Deprivation (IMD) Score',
 'Income Score (rate)',
 'Employment Score (rate)',
 'Education, Skills and Training Score',
 'Health Deprivation and Disability Score',
 'Crime Score',
 'Barriers to Housing and Services Score',
 'Living Environment Score',
 'Income Deprivation Affecting Children Index (IDACI) Score (rate)',
 'Income Deprivation Affecting Older People (IDAOPI) Score (rate)',
 'Children and Young People Sub-domain Score',
 'Adult Skills Sub-domain Score',
 'Geographical Barriers Sub-domain Score',
 'Wider Barriers Sub-domain Score',
 'Indoors Sub-domain Score',
 'Outdoors Sub-domain Score']

In [15]:
IMD5 = IMD5.rename(columns={
    'LSOA code (2021)': 'LSOA',
    'Income Score (rate)': 'income_score_rate',
    'Employment Score (rate)': 'employ_score_rate'
})

In [17]:
IMD5 = IMD5.rename(columns={
    'Living Environment Score': 'living_environment_score',
    'Barriers to Housing and Services Score': 'barriers_score'
})

In [18]:
# export edited IMD5 df
IMD5.to_parquet("../data/IMD5.parquet")

In [19]:
# merge or join the files on lsoa code

model_df_binary = model_df_binary.merge(
    IMD5[['LSOA', 'employ_score_rate', 'income_score_rate', 'living_environment_score', 'barriers_score']],
    left_on='lsoa_code',
    right_on='LSOA',
    how='left'
)

# then remove the redundant extra LSOA column 
model_df_binary = model_df_binary.drop(columns=['LSOA'])

#sanity check

model_df_binary.head()

,crime_id,month,reported_by,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,outcome_binary,employ_score_rate,income_score_rate,living_environment_score,barriers_score
0,03b3bda6de7c51f6cd0820b7f998a1f878e7acbb6b18dd...,2025-01,West Yorkshire Police,-1.882755,53.924927,On or near Moorside Lane,E01010646,Bradford 001A,Burglary,Investigation complete; no suspect identified,0,0.047,0.056,28.436,24.793
1,6f9fe9a78c7ed106fadb658e4fd5313c1bb5bc5c8aa223...,2025-01,West Yorkshire Police,-1.878940,53.943744,On or near Cross End Fold,E01010646,Bradford 001A,Shoplifting,Investigation complete; no suspect identified,0,0.047,0.056,28.436,24.793
2,9cbc69a540485581b55d8d4723eb41b97d364bc98132d8...,2025-01,West Yorkshire Police,-1.872959,53.941733,On or near Cornerstones Close,E01010646,Bradford 001A,Vehicle crime,Investigation complete; no suspect identified,0,0.047,0.056,28.436,24.793
3,050a7cc33129f37f41cbbd38fe858e63e07dc239720be8...,2025-01,West Yorkshire Police,-1.883567,53.934131,On or near Cocking Lane,E01010646,Bradford 001A,Violence and sexual offences,Unable to prosecute suspect,0,0.047,0.056,28.436,24.793
4,b43bb9c614a189dd550a9e877b409cdb6449d6ec1c1073...,2025-01,West Yorkshire Police,-1.880138,53.945417,On or near Aynholme Drive,E01010646,Bradford 001A,Violence and sexual offences,Investigation complete; no suspect identified,0,0.047,0.056,28.436,24.793


In [20]:
# check for unmatched LSOAs

unmatched = model_df_binary['employ_score_rate'].isna().sum()
print(f"Unmatched rows: {unmatched} ({unmatched/len(model_df_binary)*100:.2f}%)")

Unmatched rows: 0 (0.00%)


In [21]:
# export the cleaned binary classification file as model_df_binary

model_df_binary.to_parquet("../data/model_df_binary.parquet")